# COBOL → Java: train and evaluate on Colab

Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!git clone https://github.com/DavutKursun/cobol2java.git
%cd cobol2java
!apt-get -qq install -y gnucobol default-jdk-headless > /dev/null
!pip -q install -r requirements.txt
!cobc --version | head -1 && javac -version

## 1. Log in to Hugging Face
Create a **write** token at huggingface.co/settings/tokens. Better: store it as a Colab secret named `HF_TOKEN`.

In [ ]:
import os
from huggingface_hub import login
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass
login(token=os.environ.get('HF_TOKEN'))

## 2. Check the dataset
The repo ships only 5 seed pairs; a useful fine-tune needs a few hundred. Generate them **on your own machine** (`python scripts/generate_pairs.py --count 50`, repeated) and push `data/programs` to GitHub before running this notebook: pairs generated here are lost when the Colab runtime resets. To generate here anyway, uncomment the first line and download `data/programs` afterwards.

In [ ]:
# !python scripts/generate_pairs.py --count 50
!python scripts/verify.py data/programs
!python scripts/build_dataset.py --reverify

## 3. Baseline: evaluate the base model *before* training

In [ ]:
!python scripts/evaluate_model.py --model Qwen/Qwen2.5-Coder-1.5B-Instruct --name base

## 4. Fine-tune with LoRA

In [ ]:
!python scripts/train_lora.py --epochs 3 --merge

## 5. Evaluate the fine-tuned model

In [ ]:
!python scripts/evaluate_model.py --model outputs/cobol2java-lora/merged --name finetuned

## 6. Publish the model and dataset

In [ ]:
!python scripts/build_dataset.py --push-to-hub DavutKursun/cobol-java-verified
from transformers import AutoModelForCausalLM, AutoTokenizer
repo = 'DavutKursun/cobol2java-qwen2.5-coder-1.5b'
AutoModelForCausalLM.from_pretrained('outputs/cobol2java-lora/merged').push_to_hub(repo)
AutoTokenizer.from_pretrained('outputs/cobol2java-lora/merged').push_to_hub(repo)